# Lab 02 solution: Add a fourth tool

The worked solution to the lab's challenge, run with its outputs. It repeats the setup, the three lookups, their
schemas and the two helpers from the lab, so it runs on its own; then it adds `get_last_invoice` and asks the two
challenge questions.

## Setup and the pieces from the lab

In [1]:
from pathlib import Path
import json
import os
import re
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI()
SYSTEM = "You are a support assistant for the Chinook music shop."

# The shop database, opened read-only
DB = ROOT / "data" / "chinook.db"
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name):
    """Find customers whose first and last name match `name` exactly."""
    rows = con.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?",
        (name,),
    ).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]

In [3]:
def get_invoices(customer_id):
    """Count a customer's invoices and add up their totals."""
    count, total = con.execute(
        "SELECT COUNT(*), SUM(Total) FROM invoices WHERE CustomerId = ?", (customer_id,)
    ).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": round(total or 0, 2)}


def get_support_rep(customer_id):
    """Name the support rep assigned to a customer."""
    found = con.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?",
        (customer_id,),
    ).fetchone()
    return {"customer_id": customer_id, "support_rep": found[0] if found else None}

The three schemas, written as in the lab (strict, `additionalProperties: false`, every field required).

In [4]:
FIND_CUSTOMER_TOOL = {
    "type": "function",
    "function": {
        "name": "find_customer",
        "description": "Find a Chinook customer by full name. Returns a list of matches, each with its customer_id.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "The customer's first and last name"}},
            "required": ["name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}
GET_INVOICES_TOOL = {
    "type": "function",
    "function": {
        "name": "get_invoices",
        "description": "Count a customer's invoices and add up their total in US dollars.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer", "description": "The customer_id from find_customer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

In [5]:
GET_SUPPORT_REP_TOOL = {
    "type": "function",
    "function": {
        "name": "get_support_rep",
        "description": "Name the support rep assigned to a customer.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer", "description": "The customer_id from find_customer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

The two helpers from Part E: `ask_model` sends the list with the tools and returns the assistant message;
`run_tool_calls` appends that message, runs each call in order through `FUNCTIONS`, and appends one tool message per
call id.

In [6]:
def ask_model(messages, tools):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        reasoning_effort=EFFORT or omit,
        max_completion_tokens=300,
    )
    return response.choices[0].message


def run_tool_calls(reply, messages):
    messages.append(reply)
    for call in reply.tool_calls:
        args = json.loads(call.function.arguments)
        result = FUNCTIONS[call.function.name](**args)
        print("ran", call.function.name, args, "->", result)
        content = json.dumps(result, ensure_ascii=False)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

## Task 1: Add `get_last_invoice` and ask about the last purchase

The new function, called by hand first. `ORDER BY InvoiceDate DESC LIMIT 1` sorts the customer's invoices newest
first and keeps one row. `InvoiceDate` is stored as text with a time (`2013-11-13 00:00:00`), so `[:10]` keeps the
date.

In [7]:
def get_last_invoice(customer_id):
    """Return a customer's most recent invoice: its id, date and total."""
    found = con.execute(
        "SELECT InvoiceId, InvoiceDate, Total FROM invoices WHERE CustomerId = ? "
        "ORDER BY InvoiceDate DESC LIMIT 1",
        (customer_id,),
    ).fetchone()
    if found is None:
        return {"customer_id": customer_id, "last_invoice": None}
    return {"customer_id": customer_id, "invoice_id": found[0], "date": found[1][:10], "total": found[2]}


print(get_last_invoice(6))

{'customer_id': 6, 'invoice_id': 404, 'date': '2013-11-13', 'total': 25.86}


Invoice 404, dated 2013-11-13, for $25.86: the newest of the seven invoices that `get_invoices` counts. Now its schema, in the same strict form, and the two lists with the fourth tool added.

In [8]:
GET_LAST_INVOICE_TOOL = {
    "type": "function",
    "function": {
        "name": "get_last_invoice",
        "description": "Get a customer's most recent invoice: its invoice_id, date and total in US dollars.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer", "description": "The customer_id from find_customer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

TOOLS = [FIND_CUSTOMER_TOOL, GET_INVOICES_TOOL, GET_SUPPORT_REP_TOOL, GET_LAST_INVOICE_TOOL]
FUNCTIONS = {
    "find_customer": find_customer,
    "get_invoices": get_invoices,
    "get_support_rep": get_support_rep,
    "get_last_invoice": get_last_invoice,
}
print([tool["function"]["name"] for tool in TOOLS])

['find_customer', 'get_invoices', 'get_support_rep', 'get_last_invoice']


The question, turn by turn, as in Part E.

In [9]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "When was Helena Holý's last purchase, and how much was it?"},
]

first = ask_model(messages, TOOLS)
print([(c.function.name, c.function.arguments) for c in first.tool_calls])
run_tool_calls(first, messages)

[('find_customer', '{"name":"Helena Holý"}')]
ran find_customer {'name': 'Helena Holý'} -> [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]


In [10]:
second = ask_model(messages, TOOLS)
print([(c.function.name, c.function.arguments) for c in second.tool_calls])
run_tool_calls(second, messages)

[('get_last_invoice', '{"customer_id":6}')]
ran get_last_invoice {'customer_id': 6} -> {'customer_id': 6, 'invoice_id': 404, 'date': '2013-11-13', 'total': 25.86}


Two turns of tool calls, as expected: `find_customer` first, because the new schema needs a `customer_id`, then `get_last_invoice` with `customer_id` 6.

In [11]:
answer = ask_model(messages, TOOLS)
print("tool_calls:", answer.tool_calls)
print(answer.content)
for fact in ["2013", "25.86"]:
    print("✅" if fact in answer.content else "❌", fact)

tool_calls: None
Helena Holý’s most recent purchase was on November 13, 2013, for $25.86.
✅ 2013
✅ 25.86


The answer gives 13 November 2013 and $25.86, the values `get_last_invoice` returned. The model wrote the date in words; the check looks for `2013` and `25.86`, so it passes either way.

## Task 2: Ask for two lookups at once

A question that needs the new tool and an old one, both keyed by the same `customer_id`.

In [12]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Who is Helena Holý's support rep, and when was her last purchase?"},
]

first = ask_model(messages, TOOLS)
print([(c.function.name, c.function.arguments) for c in first.tool_calls])
run_tool_calls(first, messages)

[('find_customer', '{"name":"Helena Holý"}')]
ran find_customer {'name': 'Helena Holý'} -> [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]


In [13]:
second = ask_model(messages, TOOLS)
print([(c.function.name, c.function.arguments) for c in second.tool_calls])
run_tool_calls(second, messages)

[('get_support_rep', '{"customer_id": 6}'), ('get_last_invoice', '{"customer_id": 6}')]
ran get_support_rep {'customer_id': 6} -> {'customer_id': 6, 'support_rep': 'Steve Johnson'}
ran get_last_invoice {'customer_id': 6} -> {'customer_id': 6, 'invoice_id': 404, 'date': '2013-11-13', 'total': 25.86}


After `find_customer`, one reply with two tool calls: `get_support_rep` and `get_last_invoice`, both with `customer_id` 6. They are independent, so the model asked for both at once, and `run_tool_calls` ran them in order.

In [14]:
answer = ask_model(messages, TOOLS)
print("tool_calls:", answer.tool_calls)
print(answer.content)
for fact in ["Steve Johnson", "2013"]:
    print("✅" if fact in answer.content else "❌", fact)

tool_calls: None
Helena Holý’s support rep is **Steve Johnson**. Her most recent purchase was on **November 13, 2013**.
✅ Steve Johnson
✅ 2013


Steve Johnson and 13 November 2013. The amount is not in the reply, and that is right: the question asked only when, not how much.

The new tool needed no change to `ask_model` or `run_tool_calls`: a function, its schema in `TOOLS` and its name in
`FUNCTIONS` were enough. The model learned it exists only from the schema.